In [1]:
from importlib import reload
# from unittest.mock import inplace

import pandas as pd

import forex_factory
from Archive import trade, oanda_pricing, llm_chart_analysis
from Archive.trade import fn_modify_trade

reload(oanda_pricing)
reload(llm_chart_analysis)
reload(trade)
reload(forex_factory)

from os.path import exists
from datetime import datetime

# Tokens and URLs
gemini_token = ''

# news_token = ''
# news_url = 'https://newsapi.org/v2/everything'

# Practice Acct No:
oanda_acct = ''
oanda_token = ''
oanda_url = 'api-fxpractice.oanda.com'

path_forex_factory = ''
path_currency_schedule = ''
path_trades = ''

In [3]:
df_currency_schedule = pd.read_csv(path_currency_schedule)
current_hour = datetime.now().hour
# current_hour = 10
current_day = datetime.now().weekday()

df_currency_schedule = df_currency_schedule[
    (df_currency_schedule['start_hour'] <= current_hour) &
    (df_currency_schedule['end_hour'] >= current_hour)]

if current_day == 6 and current_hour >= 17:         # Sunday
    trading_day = True
elif 0 <= current_day <= 3:         # Monday through Thursday
    trading_day = True
elif current_day == 4 and current_hour < 17:        # Friday
    trading_day = True
else:                                               # Weekend
    trading_day = False

if trading_day:
    li_currencies = df_currency_schedule['currency'].tolist()
    print(f'Currencies currently available to trade: {li_currencies}')
else:
    print('Not a trading day')
    li_currencies = None


Currencies currently available to trade: ['GBP_NZD', 'GBP_AUD']


In [4]:
for currency in li_currencies:
    print(f'--------------- {currency} ---------------')
    # Check if currency is within trading hours
    if df_currency_schedule['start_hour'] <= current_hour < df_currency_schedule['end_hour']:
        continue
    # else:
    # skip loop iteration


    # Obtain Forex Factory dataframe
    df_forex_factory = forex_factory.fn_forex_factory(path_forex_factory=path_forex_factory)

    # Check for open orders
    df_open_orders = trade.fn_open_orders(
        oanda_token=oanda_token,
        oanda_url=oanda_url,
        oanda_acct=oanda_acct
        )

    if not df_open_orders.empty:
        li_open_orders = set(
            df_open_orders['instrument'].tolist())
    else:
        li_open_orders = []

    # Evaluate currency for a new trade
    if currency not in li_open_orders:
        print(f'No trade already open for {currency}')

        # Pull historical data
        df_oanda_historical = (
            oanda_pricing.fn_oanda_historical(
                currency=currency,
                oanda_token=oanda_token,
                oanda_url=oanda_url,
                oanda_acct=oanda_acct
        ))
        # Check current prices
        df_oanda_pricing = (
            oanda_pricing.fn_oanda_current(
                currency=currency,
                oanda_token=oanda_token,
                oanda_url=oanda_url,
                oanda_acct=oanda_acct
        ))
        # Analyze price action and news
        llm_response = (
            llm_chart_analysis.fn_llm_chart_analysis(
                currency=currency,
                df_forex_factory=df_forex_factory,
                df_oanda_historical=df_oanda_historical,
                df_oanda_pricing=df_oanda_pricing,
                gemini_token=gemini_token
        ))
        # Place trade
        end_hour = df_currency_schedule[df_currency_schedule['currency'] == currency]['end_hour'].values[0]

        df_trade = (
            trade.fn_place_trade(
                currency=currency,
                response_llm=llm_response,
                risk=0.01,
                end_hour=end_hour,
                oanda_token=oanda_token,
                oanda_url=oanda_url,
                oanda_acct=oanda_acct
        ))
        # Log trade
        df_trade['reasoning'] = llm_response
        df_trade.to_csv(path_trades,
            mode='a',
            index=False,
            header=not exists(path_trades))
    else:
        print(f'Trade already open for {currency}')

        # Pull historical data
        df_oanda_historical = (
            oanda_pricing.fn_oanda_historical(
                currency=currency,
                oanda_token=oanda_token,
                oanda_url=oanda_url,
                oanda_acct=oanda_acct
        ))
        # Check current prices
        df_oanda_pricing = (
            oanda_pricing.fn_oanda_current(
                currency=currency,
                oanda_token=oanda_token,
                oanda_url=oanda_url,
                oanda_acct=oanda_acct
        ))

        llm_response = llm_chart_analysis.fn_llm_checkin()

        trade_id = df_open_orders[df_open_orders['instrument'] == currency]['id'].values[0]
        fn_modify_trade(
            oanda_token=oanda_token,
            oanda_url=oanda_url,
            oanda_acct=oanda_acct,
            currency=currency,
            trade_id=trade_id,
            response_llm=llm_response
        )

    print('---------------------------------------')


# finish adding in check-ins/re-eval before red news
# add in valid to dates -- Implemented from currency schedule. Can LLM provide those?
# trailing stop/move to 0 loss after N pips



--------------- GBP_NZD ---------------


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [216]:
print(llm_response)

**Market Analysis:**

The GBPUSD pair has shown a clear bullish recovery over the past three trading days (August 25-27), following a brief retracement. The daily candles indicate strong buying pressure, with the price consistently closing higher. The current price action sees GBPUSD trading near the high of the most recent daily candle (August 27th), suggesting a continuation of this upward momentum.

From the historical data, the high of August 21st at 1.35444 stands out as a significant resistance level that the market could target. Immediate support can be identified around the current day's low of 1.34934 and the previous day's high of 1.35022, which the price has now broken above.

Looking at the news calendar, there are no high-impact news events for GBP or USD scheduled until Friday morning (US/Eastern) with the Revised UoM Consumer Sentiment. This absence of immediate fundamental catalysts suggests that technical factors and current market sentiment are likely to drive price a

Development

12
2025-08-29 12:59:59
12
2025-08-29 12:59:59
12
2025-08-29 12:59:59
12
2025-08-29 12:59:59
